# 🔑 Solutions: Lesson 07 · Project 1 · Handwritten Digit Recognition

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 7.1: Per-class accuracy from a confusion matrix  &nbsp; `⭐`

Given a confusion matrix `cm` (rows = true class, columns = predicted), write `per_class_accuracy(cm)` returning each class's **recall**: the diagonal divided by the row sum.


In [2]:
def per_class_accuracy(cm):
    return np.diag(cm) / cm.sum(axis=1)

In [3]:
# ▶ self-check
cm = np.array([[50, 2, 3], [5, 40, 5], [0, 1, 9]])
r = per_class_accuracy(cm)
dl.check("returns [50/55, 40/50, 9/10]", r is not None and np.allclose(r, [50/55, 0.8, 0.9]))

✅ returns [50/55, 40/50, 9/10]


---
### 🧩 Problem 7.2: Most confidently wrong  &nbsp; `⭐⭐`

Write `most_confident_mistakes(probs, y, k=5)`. `probs` is an `(N, C)` NumPy array of predicted probabilities and `y` the true labels.
Return the **indices of the `k` misclassified examples whose predicted probability was highest**, sorted from most to least confident.


In [4]:
def most_confident_mistakes(probs, y, k=5):
    pred = probs.argmax(1)
    wrong = np.where(pred != y)[0]
    conf = probs[wrong, pred[wrong]]
    return wrong[np.argsort(-conf)[:k]]

In [5]:
# ▶ self-check
probs = np.array([[.9, .1], [.2, .8], [.6, .4], [.05, .95], [.7, .3]])
y = np.array([0, 0, 1, 0, 0])
res = most_confident_mistakes(probs, y, k=2)
dl.check("returns [3, 1] (wrong examples: idx 1 at 0.8, idx 2 at 0.6, idx 3 at 0.95)", res is not None and list(res) == [3, 1])
dl.check("never returns correct predictions", res is not None and all(np.argmax(probs[i]) != y[i] for i in res))

✅ returns [3, 1] (wrong examples: idx 1 at 0.8, idx 2 at 0.6, idx 3 at 0.95)
✅ never returns correct predictions


---
### 🧩 Problem 7.3: Translate an image  &nbsp; `⭐⭐`

Implement `shift_images(x, dx, dy=0)`: translate each image in the batch `x` (shape `(N, H, W)`) right by `dx` pixels and down by `dy` pixels (negative = left/up),
filling vacated pixels with **0** (no wrap-around!).


In [6]:
def shift_images(x, dx, dy=0):
    out = np.zeros_like(x); H, W = x.shape[1:]
    sy, ty = slice(max(0, -dy), H - max(0, dy)), slice(max(0, dy), H - max(0, -dy))
    sx, tx = slice(max(0, -dx), W - max(0, dx)), slice(max(0, dx), W - max(0, -dx))
    out[:, ty, tx] = x[:, sy, sx]
    return out

In [7]:
# ▶ self-check
x = np.arange(1, 10, dtype=float).reshape(1, 3, 3)       # [[1 2 3],[4 5 6],[7 8 9]]
r = shift_images(x, 1, 0)
dl.check("shift right by 1", r is not None and np.array_equal(r[0], [[0, 1, 2], [0, 4, 5], [0, 7, 8]]))
dl.check("shift left by 1", np.array_equal(shift_images(x, -1)[0], [[2, 3, 0], [5, 6, 0], [8, 9, 0]]))
dl.check("shift down by 1", np.array_equal(shift_images(x, 0, 1)[0], [[0, 0, 0], [1, 2, 3], [4, 5, 6]]))
dl.check("no shift returns the same image", np.array_equal(shift_images(x, 0)[0], x[0]))

✅ shift right by 1
✅ shift left by 1
✅ shift down by 1
✅ no shift returns the same image


---
### 🧩 Problem 7.4: Reject option: coverage vs accuracy  &nbsp; `⭐⭐`

Write `reject_curve(probs, y, thresholds)` returning a list of `(threshold, coverage, accuracy_on_kept)` tuples, where *coverage* is the fraction of examples with max-probability ≥ threshold
and the accuracy is measured **only on those kept examples** (use `float('nan')` if none are kept).


In [8]:
def reject_curve(probs, y, thresholds):
    conf, pred, out = probs.max(1), probs.argmax(1), []
    for t in thresholds:
        keep = conf >= t
        acc = (pred[keep] == y[keep]).mean() if keep.any() else float("nan")
        out.append((t, keep.mean(), acc))
    return out

In [9]:
# ▶ self-check
probs = np.array([[.9, .1], [.6, .4], [.55, .45], [.2, .8]]); y = np.array([0, 1, 0, 1])
r = reject_curve(probs, y, [0.5, 0.7, 0.95])
ok = r is not None and len(r) == 3
dl.check("threshold .5 keeps everyone, accuracy 3/4", ok and np.isclose(r[0][1], 1.0) and np.isclose(r[0][2], 0.75))
dl.check("threshold .7 keeps 2 of 4, both right", ok and np.isclose(r[1][1], 0.5) and np.isclose(r[1][2], 1.0))
dl.check("threshold .95 keeps nobody → NaN accuracy", ok and r[2][1] == 0 and np.isnan(r[2][2]))

✅ threshold .5 keeps everyone, accuracy 3/4
✅ threshold .7 keeps 2 of 4, both right
✅ threshold .95 keeps nobody → NaN accuracy


**Why this works.** Raising the threshold trades coverage for reliability: exactly the dial a real deployment tunes.

---
### 🧩 Problem 7.5: Design challenge: a lean, accurate model  &nbsp; `⭐⭐⭐`

Return an `nn.Module` from `my_model()` with **at most 120,000 parameters** that, under our fixed training protocol
(16 000 training images, Adam lr 1e-3, batch 128, **8 epochs**), reaches **≥ 95.5 % validation accuracy**.
The starter is plain logistic regression (≈ 91 %). Improve it! (The self-check runs the protocol for you; about 15 seconds.)


In [10]:
def my_model():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))

In [11]:
# ▶ self-check
import torch
from torch.utils.data import TensorDataset, DataLoader
xa, ya, _, _ = dl.load_mnist()
perm = np.random.default_rng(0).permutation(len(ya)); vi, ti = perm[:4000], perm[4000:]
m_, s_ = xa[ti].mean(), xa[ti].std()
P = lambda a: torch.tensor((a - m_) / s_, dtype=torch.float32)
tdl = DataLoader(TensorDataset(P(xa[ti]), torch.tensor(ya[ti])), batch_size=128, shuffle=True)
vdl = DataLoader(TensorDataset(P(xa[vi]), torch.tensor(ya[vi])), batch_size=1000)
dl.set_seed(0)
net = my_model()
n = dl.count_params(net)
dl.check(f"≤ 120,000 parameters (yours: {n:,})", n <= 120_000)
h = dl.fit(net, tdl, vdl, torch.nn.CrossEntropyLoss(), torch.optim.Adam(net.parameters(), lr=1e-3), epochs=8, device=torch.device("cpu"), verbose=False)
dl.check(f"validation accuracy ≥ 95.5% (yours: {h['val_acc'][-1]:.2%})", h["val_acc"][-1] >= 0.955, "add a hidden layer with ReLU")

✅ ≤ 120,000 parameters (yours: 109,386)


✅ validation accuracy ≥ 95.5% (yours: 96.05%)


**Why this works.** A small hidden layer lets the network build intermediate 'stroke detectors'; logistic regression can only weigh raw pixels. Adding *capacity in the right place* beats adding parameters blindly.